In [2]:
pip install tqdm

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import pandas as pd
from tqdm import tqdm
import time
import random
from googleapiclient.errors import HttpError

from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from google.auth.transport.requests import Request

SCOPES = ["https://www.googleapis.com/auth/drive.readonly"]
FOLDER_MIME = "application/vnd.google-apps.folder"


def get_drive_service():
    creds = None

    # 이미 로그인한 토큰이 있으면 재사용
    if os.path.exists("token.json"):
        creds = Credentials.from_authorized_user_file("token.json", SCOPES)

    # 토큰이 없거나 만료됐으면 새로 로그인
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file("credentials.json", SCOPES)
            creds = flow.run_local_server(port=0)

        # 다음 실행을 위해 토큰 저장
        with open("token.json", "w") as token:
            token.write(creds.to_json())

    service = build("drive", "v3", credentials=creds)
    return service


def execute_with_backoff(request, max_retries=8, base_sleep=1.0):
    """
    403 userRateLimitExceeded / 429 등에서 지수 백오프 재시도
    """
    for attempt in range(max_retries):
        try:
            return request.execute()
        except HttpError as e:
            status = getattr(e.resp, "status", None)
            content = str(e)
            retriable = (status in [403, 429, 500, 503]) and (
                "userRateLimitExceeded" in content or "rate limit" in content.lower() or status != 403
            )
            if not retriable or attempt == max_retries - 1:
                raise

            sleep = base_sleep * (2 ** attempt) + random.uniform(0, 0.5)
            print(f"[backoff] HTTP {status} rate-limited. sleep {sleep:.2f}s (attempt {attempt+1}/{max_retries})")
            time.sleep(sleep)


def list_all_files(service):
    items = []
    page_token = None
    pbar = tqdm(desc="Listing files/folders (pages)", unit="page")

    while True:
        req = service.files().list(
            q="trashed = false",
            pageSize=1000,
            pageToken=page_token,
            fields="nextPageToken, files(id,name,mimeType,parents,driveId,modifiedTime,createdTime)",
            includeItemsFromAllDrives=True,
            supportsAllDrives=True,
            corpora="allDrives",
        )
        resp = execute_with_backoff(req)

        batch = resp.get("files", [])
        items.extend(batch)

        page_token = resp.get("nextPageToken")
        pbar.update(1)
        pbar.set_postfix(items=len(items))

        time.sleep(0.15)

        if not page_token:
            break

    pbar.close()
    return items


def list_shared_drives(service):
    drive_map = {}
    page_token = None
    pbar = tqdm(desc="Listing shared drives (pages)", unit="page")

    while True:
        req = service.drives().list(
            pageSize=100,
            pageToken=page_token,
            fields="nextPageToken, drives(id,name)"
        )
        resp = execute_with_backoff(req)

        for d in resp.get("drives", []):
            drive_map[d["id"]] = d["name"]

        page_token = resp.get("nextPageToken")
        pbar.update(1)

        if not page_token:
            break

    pbar.close()
    return drive_map


def build_full_paths(items, shared_drive_name_by_id):
    item_by_id = {it["id"]: it for it in items}
    path_cache = {}

    def drive_label(it):
        did = it.get("driveId")
        if did and did in shared_drive_name_by_id:
            return shared_drive_name_by_id[did]
        return "My Drive"

    def full_path(file_id, stack=None):
        if file_id in path_cache:
            return path_cache[file_id]

        it = item_by_id.get(file_id)
        if not it:
            return ""

        if stack is None:
            stack = set()
        if file_id in stack:
            return f"/{drive_label(it)}/[CYCLE]/{it.get('name', '')}"

        stack.add(file_id)

        name = it.get("name", "")
        parents = it.get("parents", [])

        if not parents:
            p = f"/{drive_label(it)}/{name}"
            path_cache[file_id] = p
            stack.remove(file_id)
            return p

        parent_id = parents[0]
        parent_it = item_by_id.get(parent_id)

        if not parent_it:
            p = f"/{drive_label(it)}/{name}"
            path_cache[file_id] = p
            stack.remove(file_id)
            return p

        parent_path = full_path(parent_id, stack)
        p = f"{parent_path}/{name}"
        path_cache[file_id] = p
        stack.remove(file_id)
        return p

    out = {}
    for it in tqdm(items, desc="Building full_path", unit="item"):
        fid = it["id"]
        out[fid] = full_path(fid)

    return out


def main():
    service = get_drive_service()

    items = list_all_files(service)
    shared_drive_name_by_id = list_shared_drives(service)
    path_by_id = build_full_paths(items, shared_drive_name_by_id)

    rows = []
    for it in tqdm(items, desc="Preparing rows", unit="item"):
        fid = it.get("id")
        mime = it.get("mimeType")
        is_folder = (mime == FOLDER_MIME)
        parents = it.get("parents", [])

        rows.append({
            "id": fid,
            "name": it.get("name", ""),
            "type": "folder" if is_folder else "file",
            "mimeType": mime,
            "driveId": it.get("driveId", ""),
            "driveName": shared_drive_name_by_id.get(it.get("driveId", ""), "My Drive"),
            "parent_id": parents[0] if parents else "",
            "full_path": path_by_id.get(fid, ""),
            "createdTime": it.get("createdTime", ""),
            "modifiedTime": it.get("modifiedTime", ""),
        })

    df = pd.DataFrame(rows)

    tqdm.write("Saving CSV: drive_inventory_with_full_path.csv")
    df.to_csv("drive_inventory_with_full_path.csv", index=False, encoding="utf-8-sig")
    tqdm.write(f"Done. rows={len(df)}")


if __name__ == "__main__":
    main()

Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=749941011301-d1q3a5v2a22dtdpb55q4diau53fhguh1.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A52370%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive.readonly&state=4ck4HkD9mGhW4Nctq4hZyVEqgGJ2jB&access_type=offline


Listing files/folders (pages): 211page [05:14,  1.49s/page, items=96953]
Listing shared drives (pages): 1page [00:00,  2.27page/s]
Preparing rows: 100%|██████████████████████████████████████████████████████| 96953/96953 [00:00<00:00, 451627.49item/s]


Saving CSV: drive_inventory_with_full_path.csv
Done. rows=96953


In [1]:
import re
import pandas as pd
from datetime import datetime, timezone

DATE_YYYYMMDD = re.compile(r"\b(20\d{2})(0[1-9]|1[0-2])(0[1-9]|[12]\d|3[01])\b")
DATE_YYMMDD   = re.compile(r"\b(\d{2})(0[1-9]|1[0-2])(0[1-9]|[12]\d|3[01])\b")

def iso_to_dt(s: str):
    # '2026-01-28T07:17:33.332Z' 형태 가정
    return datetime.fromisoformat(s.replace("Z", "+00:00"))

def normalize_spaces(name: str) -> str:
    name = name.strip()
    name = re.sub(r"\s+", " ", name)
    return name

def detect_issues(row) -> list[dict]:
    issues = []
    name = str(row["name"])
    t = str(row["type"])

    # 1) 앞뒤 공백 / 연속 공백
    if name != name.strip():
        issues.append({"type": "TRIM_NEEDED", "severity": 2, "detail": "leading/trailing spaces"})
    if re.search(r"\s{2,}", name):
        issues.append({"type": "MULTI_SPACE", "severity": 1, "detail": "multiple consecutive spaces"})

    # 2) 구분자 혼용(간단 휴리스틱)
    if "_" in name and "-" in name:
        issues.append({"type": "MIXED_SEPARATORS", "severity": 2, "detail": "contains both '_' and '-'"})

    # 3) 열린 범위 표기(~) 감지
    if name.endswith("~"):
        issues.append({"type": "TRAILING_TILDE", "severity": 3, "detail": "name ends with '~' (open-ended range?)"})

    # 4) 날짜 토큰 추출 + 메타데이터 날짜와의 거리
    created = iso_to_dt(row["createdTime"]) if pd.notna(row["createdTime"]) else None
    m = DATE_YYYYMMDD.search(name) or DATE_YYMMDD.search(name)
    if m and created:
        token = m.group(0)
        # YYMMDD면 20YY로 가정(업무 데이터면 보통 20xx)
        if len(token) == 6:
            y = int(token[:2]) + 2000
            token_dt = datetime(y, int(token[2:4]), int(token[4:6]), tzinfo=timezone.utc)
        else:
            token_dt = datetime(int(token[:4]), int(token[4:6]), int(token[6:8]), tzinfo=timezone.utc)

        # createdTime과 90일 이상 차이면 후보로 올림(조직 규칙에 맞게 조정)
        delta_days = abs((created - token_dt).days)
        if delta_days >= 90:
            issues.append({
                "type": "DATE_TOKEN_MISMATCH",
                "severity": 2,
                "detail": f"date token {token} far from createdTime ({delta_days}d)"
            })

    # 5) mimeType 기반 확장자 체크(파일만)
    mime = str(row.get("mimeType", ""))
    if t == "file":
        if mime == "application/haansofthwp" and not name.lower().endswith(".hwp"):
            issues.append({"type": "EXT_MISMATCH", "severity": 2, "detail": "HWP mime but filename not .hwp"})
        if "spreadsheet" in mime and name.lower().endswith(".xlsx"):
            # 구글시트인데 xlsx 확장자면 혼동 가능(반대로도 체크 가능)
            issues.append({"type": "TYPE_CONFUSION", "severity": 1, "detail": "Google Sheet but name looks like .xlsx"})

    return issues

def suggest_name(row, issues) -> str:
    # 최소한의 안전한 제안: 공백 정리 + 일부 치명적 패턴만 수정
    name = str(row["name"])
    s = normalize_spaces(name)

    if any(i["type"] == "TRAILING_TILDE" for i in issues):
        # 예: _202505~ -> _202505- (또는 _202505_onwards 등 조직 규칙 필요)
        s = re.sub(r"~$", "", s).rstrip()

    # 예: 괄호 앞뒤 공백 정리 "( 기술... )" 같은 것
    s = re.sub(r"\(\s+", "(", s)
    s = re.sub(r"\s+\)", ")", s)
    s = re.sub(r"\s*&\s*", "&", s)  # & 주변 공백 통일(원하면 제거)

    return s

def run_audit(csv_path: str, out_report: str = "report.csv"):
    df = pd.read_csv(csv_path)
    rows = []
    for _, r in df.iterrows():
        issues = detect_issues(r)
        severity = sum(i["severity"] for i in issues)
        suggested = suggest_name(r, issues) if issues else ""
        rows.append({
            "driveName": r.get("driveName", ""),
            "full_path": r.get("full_path", ""),
            "type": r.get("type", ""),
            "name": r.get("name", ""),
            "severity": severity,
            "issues": ";".join([i["type"] for i in issues]),
            "suggested_name": suggested,
            "createdTime": r.get("createdTime", ""),
            "modifiedTime": r.get("modifiedTime", ""),
        })
    rep = pd.DataFrame(rows).sort_values(["severity", "full_path"], ascending=[False, True])
    rep.to_csv(out_report, index=False, encoding="utf-8-sig")
    return rep

# 사용 예:
# rep = run_audit("drive_inventory.csv", "naming_report.csv")


In [20]:
a = pd.read_csv('drive_inventory_with_full_path.csv')
a_df = a[:900]
a_df.to_csv('drive_inventory.csv', encoding='utf-8')

In [9]:
rep = run_audit("drive_inventory.csv", "naming_report.csv")

In [5]:
!pip install -U python-dotenv
from dotenv import load_dotenv
load_dotenv()  # 현재 작업 폴더의 .env 읽음

import os
print(os.getenv("HF_TOKEN")[:10], "...")


hf_PsFiXjs ...


In [22]:
import requests, json

BASE_URL = "http://127.0.0.1:1234/v1"
r = requests.get(f"{BASE_URL}/models", timeout=10)
print(r.status_code)
print([m["id"] for m in r.json()["data"]])


200
['zai-org/glm-4.6v-flash', 'text-embedding-nomic-embed-text-v1.5']


In [3]:
import os, json, re, sys, time
import pandas as pd
import requests
from tqdm.auto import tqdm

CSV_PATH = "drive_inventory.csv"
OUT_PATH = "rename_plan.csv"

# ✅ 중간 저장 파일
PARTIAL_PATH = "rename_plan_partial.csv"
CKPT_PATH = "rename_checkpoint.json"

BASE_URL = "http://127.0.0.1:1234/v1"
MODEL_ID = "zai-org/glm-4.6v-flash"

STANDARD = {
    "date_rule": "날짜 표기는 YYMMDD 또는 YYYYMMDD 중 하나로 통일(혼용 금지)",
    "separator_rule": "구분자는 '_' 중심으로 일관(공백/하이픈 혼용 최소화)",
    "no_new_entities": "원문에 없는 기관명/고유명사 추가 금지",
    "no_content_guess": "파일 내용을 추측하지 말 것(파일명 문자열에서만 재구성)"
}

# ✅ Session 재사용
SESSION = requests.Session()
SESSION.headers.update({"Content-Type": "application/json", "Authorization": "Bearer lm-studio"})

# -------------------------
# Checkpoint helpers
# -------------------------
def load_ckpt() -> dict:
    if os.path.exists(CKPT_PATH):
        with open(CKPT_PATH, "r", encoding="utf-8") as f:
            return json.load(f)
    return {
        "done_folders": [],          # 완료된 folder_path 리스트
        "stats": {
            "folders_done": 0,
            "files_done": 0,
            "last_folder": "",
            "updated_at": ""
        },
        "version": 1
    }

import os, json, time

def save_ckpt(ckpt: dict, retries: int = 10, backoff_sec: float = 0.12):
    """
    - 현재 작업 폴더(CKPT_PATH) 그대로 사용
    - Windows에서 파일 잠금(WinError 5) 발생 시 재시도
    - 그래도 실패하면 atomic replace 대신 직접 덮어쓰기(fallback)
    """
    ckpt["stats"]["updated_at"] = time.strftime("%Y-%m-%d %H:%M:%S")

    tmp = CKPT_PATH + ".tmp"

    # 1) tmp에 먼저 쓰기
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(ckpt, f, ensure_ascii=False, indent=2)

    # 2) os.replace 재시도 (atomic)
    last_err = None
    for i in range(retries):
        try:
            os.replace(tmp, CKPT_PATH)
            return
        except PermissionError as e:
            last_err = e
            time.sleep(backoff_sec * (i + 1))

    # 3) fallback: 직접 덮어쓰기 (atomic은 아니지만 체크포인트 용도로 충분)
    try:
        with open(CKPT_PATH, "w", encoding="utf-8") as f:
            json.dump(ckpt, f, ensure_ascii=False, indent=2)
        try:
            os.remove(tmp)
        except OSError:
            pass
        return
    except Exception as e:
        raise PermissionError(
            f"Checkpoint write failed. Close any app that may be using '{CKPT_PATH}' "
            f"(editor/preview/sync/antivirus). Last PermissionError: {last_err}. "
            f"Fallback error: {type(e).__name__}: {e}"
        )

def append_rows_partial(rows: list[dict]):
    if not rows:
        return
    df = pd.DataFrame(rows)
    write_header = not os.path.exists(PARTIAL_PATH) or os.path.getsize(PARTIAL_PATH) == 0
    df.to_csv(PARTIAL_PATH, index=False, mode="a", header=write_header, encoding="utf-8-sig")

def load_done_folders_from_partial() -> set:
    """partial csv가 이미 있으면, 거기서 folder_path를 읽어 done_folders 보정 가능"""
    if not os.path.exists(PARTIAL_PATH) or os.path.getsize(PARTIAL_PATH) == 0:
        return set()
    try:
        dfp = pd.read_csv(PARTIAL_PATH, usecols=["folder_path"])
        return set(dfp["folder_path"].astype(str).tolist())
    except Exception:
        return set()

# -------------------------
# LLM helpers
# -------------------------
def assert_lmstudio_alive():
    r = SESSION.get(f"{BASE_URL}/models", timeout=(2, 5))
    r.raise_for_status()
    ids = [m.get("id") for m in r.json().get("data", [])]
    if MODEL_ID not in ids:
        raise RuntimeError(f"MODEL_ID '{MODEL_ID}' not in /v1/models. Available: {ids}")

def chat_completion(messages, temperature=0, max_tokens=250, retries=3) -> str:
    url = f"{BASE_URL}/chat/completions"
    payload = {
        "model": MODEL_ID,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
    }

    last_err = None
    for i in range(retries):
        try:
            # connect 5초, read 600초(10분)
            r = SESSION.post(url, json=payload, timeout=(5, 600))
            r.raise_for_status()
            return r.json()["choices"][0]["message"]["content"]
        except requests.exceptions.ReadTimeout as e:
            last_err = e
            time.sleep(0.5 * (i + 1))

    # retries 모두 실패했을 때
    raise last_err

THINK_RE = re.compile(r"<think>.*?</think>", flags=re.S)

def extract_json(text: str) -> dict:
    t = (text or "").strip()

    # 1) <think>...</think> 제거 (일부 모델이 reasoning을 이런 태그로 내보냄)
    t = THINK_RE.sub("", t).strip()

    # 2) JSON 구간만 잘라서 파싱 (앞뒤 잡텍스트가 있어도 OK)
    i = t.find("{")
    j = t.rfind("}")
    if i == -1 or j == -1 or j < i:
        raise ValueError(f"JSON not found in: {t[:200]}")
    candidate = t[i:j+1]

    return json.loads(candidate)

# -------------------------
# Name rules
# -------------------------
def get_folder_path(full_path: str) -> str:
    p = str(full_path or "").strip()
    if not p:
        return ""
    parts = p.split("/")
    if len(parts) <= 2:
        return p.rstrip("/")
    return "/".join(parts[:-1]).rstrip("/")

def normalize_spaces(s: str) -> str:
    return re.sub(r"\s+", " ", s).strip()

def is_candidate(name: str) -> bool:
    s = str(name)
    if not s:
        return False
    return (
        s.endswith("~")
        or len(s) >= 80
        or (("_" in s) and ("-" in s))
        or ("  " in s)
        or ("\t" in s)
        or ("\n" in s)
    )

def llm_extract_folder_rules(folder_path: str, names: list[str]) -> dict:
    names = [str(x) for x in names if str(x).strip()][:60]
    prompt_obj = {
        "standard": STANDARD,
        "folder_path": folder_path,
        "file_names_in_folder": names,
        "output_schema": {
            "primary_separator": "one of ['_','-','space']",
            "date_format": "one of ['YYYYMMDD','YYMMDD','NONE','MIXED']",
            "preferred_case": "one of ['keep','lower','upper']",
            "token_order_hint": "string (optional). ex: 'team_topic_date' or ''",
            "rewrite_rules": ["string rules"],
            "exceptions_regex": ["string regex patterns that should NOT be renamed (optional)"],
            "confidence": "0~1",
            "notes": "string[]"
        },
        "instructions": [
            "반드시 JSON 객체 1개만 출력하라. 다른 텍스트는 출력하지 말 것.",
            "파일명 리스트(문자열)만 보고 규칙을 추출하라. 내용 추측 금지.",
            "확신이 낮으면 confidence를 낮추고 notes에 'AMBIGUOUS_RULE'을 포함해라.",
            "exceptions_regex는 '그대로 두는게 좋은' 패턴만 최소한으로 제시해라.",
            "notes는 5개 이하, 각 note는 80자 이하로 제한해라."
        ]
    }
    messages = [
        {"role": "system", "content": "Return ONLY one valid JSON object. No markdown. No extra text."},
        {"role": "user", "content": json.dumps(prompt_obj, ensure_ascii=False)}
    ]
    return extract_json(chat_completion(messages, temperature=0))

# ✅ 당신 코드의 apply_folder_rules 그대로 붙여넣으세요.
def apply_folder_rules(name: str, rules: dict):
    issues, rationale = [], []
    conf = float(rules.get("confidence", 0.5) or 0.5)
    needs_human_check = False

    original = str(name)
    s2 = normalize_spaces(original)

    for pat in (rules.get("exceptions_regex") or []):
        if pat:
            try:
                if re.search(pat, s2):
                    return original, ["EXCEPTION_MATCH"], conf, True, [f"matched exceptions_regex: {pat}"]
            except re.error:
                issues.append("BAD_EXCEPTIONS_REGEX")
                needs_human_check = True

    sep = rules.get("primary_separator", "_")
    sep_char = " " if sep == "space" else ("-" if sep == "-" else "_")

    before = s2
    if sep_char == "_":
        s2 = s2.replace("-", "_")
        s2 = re.sub(r"\s+", "_", s2)
        s2 = re.sub(r"_+", "_", s2)
    elif sep_char == "-":
        s2 = s2.replace("_", "-")
        s2 = re.sub(r"\s+", "-", s2)
        s2 = re.sub(r"-+", "-", s2)
    else:
        s2 = s2.replace("_", " ").replace("-", " ")
        s2 = normalize_spaces(s2)

    if s2 != before:
        rationale.append(f"normalized separators to '{sep_char}'")

    date_fmt = rules.get("date_format", "NONE")
    if date_fmt in ("YYYYMMDD", "YYMMDD"):
        def _to_yyyymmdd(m): return f"{m.group(1)}{m.group(2)}{m.group(3)}"
        def _to_yymmdd(m):   return f"{m.group(1)[-2:]}{m.group(2)}{m.group(3)}"
        before = s2
        s2 = re.sub(
            r"\b(20\d{2})[.\-_ ]?(0\d|1[0-2])[.\-_ ]?(0\d|[12]\d|3[01])\b",
            _to_yyyymmdd if date_fmt == "YYYYMMDD" else _to_yymmdd,
            s2
        )
        if s2 != before:
            rationale.append(f"normalized date to {date_fmt}")

    case = rules.get("preferred_case", "keep")
    if case == "lower":
        s2 = s2.lower(); rationale.append("forced lowercase")
    elif case == "upper":
        s2 = s2.upper(); rationale.append("forced uppercase")

    if s2 == original:
        return original, ["NO_CHANGE"], conf, False, ["rules produced no change"]

    if conf < 0.5:
        issues.append("LOW_RULE_CONFIDENCE")
        needs_human_check = True

    return s2, issues, conf, needs_human_check, rationale

# -------------------------
# Main with resume
# -------------------------
def main():
    assert_lmstudio_alive()

    df = pd.read_csv(CSV_PATH)
    df["folder_path"] = df["full_path"].astype(str).apply(get_folder_path)

    # ✅ 후보 마스크 1회 계산
    name_s = df["name"].astype(str)
    cand_mask = name_s.map(is_candidate)

    # ✅ ckpt/partial 기반 done_folders 산출
    ckpt = load_ckpt()
    done_from_ckpt = set(map(str, ckpt.get("done_folders", [])))
    done_from_partial = load_done_folders_from_partial()
    done_folders = done_from_ckpt | done_from_partial

    total_cands = int(cand_mask.sum())
    out_count_written = 0

    # 진행바
    p_folder = tqdm(df.groupby("folder_path", dropna=False), desc="Folder rules", unit="folder")
    p_file = tqdm(total=total_cands, desc="Rename candidates", unit="file")

    # ✅ 재개 시 partial에 이미 기록된 파일 수만큼 p_file 초기 업데이트(대략치)
    if os.path.exists(PARTIAL_PATH) and os.path.getsize(PARTIAL_PATH) > 0:
        try:
            prev_rows = sum(1 for _ in open(PARTIAL_PATH, "r", encoding="utf-8-sig")) - 1
            if prev_rows > 0:
                p_file.update(min(prev_rows, total_cands))
        except Exception:
            pass

    try:
        for folder_path, g in p_folder:
            folder_path = str(folder_path)

            # ✅ 이미 완료된 폴더면 스킵
            if folder_path in done_folders:
                continue

            g = g.copy()
            names_all = g["name"].astype(str).tolist()

            # ✅ 이 폴더에서 후보만 추출 (cand_mask 재사용)
            cands = g.loc[cand_mask.reindex(g.index, fill_value=False)]
            if cands.empty:
                # 후보가 없으면 굳이 done 처리할지 선택: 여기서는 done 처리(재개 시 반복 방지)
                done_folders.add(folder_path)
                ckpt["done_folders"] = sorted(done_folders)
                ckpt["stats"]["folders_done"] = len(done_folders)
                ckpt["stats"]["last_folder"] = folder_path
                save_ckpt(ckpt)
                continue

            p_folder.set_postfix_str(f"{folder_path} | cands={len(cands)}")

            # (1) 폴더 규칙 추출 1회
            try:
                rules = llm_extract_folder_rules(folder_path, names_all)
            except Exception as e:
                rows = []
                for _, r in cands.iterrows():
                    rows.append({
                        "folder_path": folder_path,
                        "parent_id": r.get("parent_id", ""),
                        "full_path": r.get("full_path", ""),
                        "name": r.get("name", ""),
                        "keep_or_rename": "keep",
                        "suggested_name": "",
                        "issues": "FOLDER_RULES_FAILED",
                        "confidence": 0.0,
                        "needs_human_check": True,
                        "rationale": f"{type(e).__name__}: {str(e)[:160]}",
                        "approved": ""
                    })
                append_rows_partial(rows)
                out_count_written += len(rows)
                p_file.update(len(rows))

                # ✅ 폴더 완료 처리 + 체크포인트 저장
                done_folders.add(folder_path)
                ckpt["done_folders"] = sorted(done_folders)
                ckpt["stats"]["folders_done"] = len(done_folders)
                ckpt["stats"]["files_done"] += len(rows)
                ckpt["stats"]["last_folder"] = folder_path
                save_ckpt(ckpt)
                continue

            # (2) 후보 파일들에 규칙 적용
            rows = []
            for _, r in cands.iterrows():
                name = str(r.get("name", ""))
                suggested, issues, conf, needs_human, rationale = apply_folder_rules(name, rules)
                keep_or_rename = "rename" if suggested != name else "keep"

                rows.append({
                    "folder_path": folder_path,
                    "parent_id": r.get("parent_id", ""),
                    "full_path": r.get("full_path", ""),
                    "name": name,
                    "keep_or_rename": keep_or_rename,
                    "suggested_name": suggested if keep_or_rename == "rename" else "",
                    "issues": ",".join(issues),
                    "confidence": conf,
                    "needs_human_check": needs_human,
                    "rationale": " | ".join(rationale),
                    "approved": ""
                })

            # ✅ 폴더 단위로 중간 저장
            append_rows_partial(rows)
            out_count_written += len(rows)
            p_file.update(len(rows))

            # ✅ 폴더 완료 처리 + 체크포인트 저장
            done_folders.add(folder_path)
            ckpt["done_folders"] = sorted(done_folders)
            ckpt["stats"]["folders_done"] = len(done_folders)
            ckpt["stats"]["files_done"] += len(rows)
            ckpt["stats"]["last_folder"] = folder_path
            save_ckpt(ckpt)

    except KeyboardInterrupt:
        # ✅ Ctrl+C로 중단 시: 체크포인트는 이미 폴더 단위로 저장됨
        print("\n[INTERRUPTED] Saved progress. You can rerun to resume.")
    finally:
        p_file.close()

    # ✅ 최종 정리: partial -> OUT_PATH (정렬해서 생성)
    if os.path.exists(PARTIAL_PATH) and os.path.getsize(PARTIAL_PATH) > 0:
        out = pd.read_csv(PARTIAL_PATH)
        if not out.empty:
            out = out.sort_values(["needs_human_check", "confidence"], ascending=[False, True])
        out.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")
        print(f"Wrote {OUT_PATH} from partial, rows={len(out)}")
    else:
        print("No partial results found; nothing to finalize.")

if __name__ == "__main__":
    main()


Folder rules:   0%|          | 0/30 [00:00<?, ?folder/s]

Rename candidates:   0%|          | 0/3 [00:00<?, ?file/s]

Wrote rename_plan.csv from partial, rows=3


In [4]:
import os
for p in ["rename_plan_partial.csv", "rename_checkpoint.json", "rename_plan.csv"]:
    if os.path.exists(p):
        os.remove(p)
print("cleared")

cleared
